# COCO noun–patch routing — interface smoke, T4×2
**Internet ON · T4×2 · Save & Run All.** Người dùng tự chạy.
Chỉ kiểm interface/gradient trên **4 ảnh TRAIN**; mỗi GPU hai ảnh, không DDP collectives.
**0 optimizer updates, không huấn luyện, không CIDEr/CHAIR, không đánh giá dev/test.**
Dùng checkpoint original SCST-object đã khóa, không dùng checkpoint A-end/CSA/GV.
Giữ nguyên forward baseline. Quan sát attention bằng phép tính Q/K dùng chung trọng số;
đối chiếu với PyTorch MHA, kiểm logits/greedy parity, mask, causal offset và gradients.
Không chuẩn hóa lại Q slice hoặc patch slice sau khi bỏ legacy/CLS.
Box chỉ là mask cho kiểm số học; chưa phải nhãn grounding sạch hay bằng chứng gain.

In [ ]:
print('COCO ROUTING INTERFACE SMOKE: START',flush=True)
from pathlib import Path
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/qformer-scst-object-w05-legacy/qformer_scst_object_w05_legacy.pth')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
OUTPUT=Path('/kaggle/working/coco_routing_interface_smoke')
for p in (CHECKPOINT,LEGACY_PROMPT,COCO_JSON):
    assert p.is_file(),p
assert VISUAL_CACHE.is_dir(),VISUAL_CACHE
assert not OUTPUT.exists(),'Preserve existing output; use fresh session or OUTPUT.'
OUTPUT.mkdir(parents=True)


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-coco-routing-interface-'))/'repo'
LOG = Path('/tmp/upt_hidden_coco-routing-interface.log')
PROJECT_COMMIT = '6acd3c9f0553d4d12d46684bbe7dcb8c38033d6b'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('eNrtfQtz28bV6F9B05kL0oYgYPGWgs74c5020yR2Yufr3CtrMCAISqhIgAZAS4qb/37P2QewC0AUpdht2pE8IxHY3bNnz54976U/fVVXuzZPtnW1yM3t7Vcn2lfv6b/XiyavP+ZaflM0bVFeaPnHdH20qZa5ltVV0xylbZuXbVGVp1qZf8xrrc636zTLtaJtNAC63bVaVWvpcqlt0zrd5G1eNyYD/r4sNtuqbrVN2l52D21VZ/C0qqsN+2yWpcbbVrsyw8nStZY22jfvy/flT+9ef2doL+C3Fmt2fuQZ+NvHpvflMl9pHYYzQHu3zg3twy6vbw3tKodfsJxdTj8nW0ASlphs0uYq/qEq4TWMLeXnZ8+urtP6opmfvC81+ClWGgNqtnValEggWGxZteL1Im2zy2RV1E07aEk+XH1MGiBIkm8W+TJZFhsOFH8AWpNr/4vIvarrqp7pb3BvtGa3RUI0WntZNNoibfJ1UQKxS7oxhkbnO6LzGVr+YQebBYDzsoH1a9//9YVWletbU58P0V8UaZNcaQATUcTFIrpy48c7GoFoyS95XSVIq70r+LnMb7Z51uZLYCegFxK9QdS7ZXRbJWPY7cEQAbYV5kXezvSiSbJ016Rr3fgmXTf5fC8mL1XW1TY72J7LFPi8rDQG53iTb6r6tp/9VFvmGfB9rQGyK2kwYLUr22qXXeZLGW/KZWYJ5P9D7FB8c+WRcp7ZXKbb/A8xttGPh1IQ97Io8XTRYd3E18WyvYz53nScdapd5umyEe/L3SahL2BRxSamY46P6RsGBXdcdC5KlAv/SPAda/0Qf2PifqX1jB2lQc/rvLi4bM9OKNxzg+4XEAQBILXocw6bRN+Ibhz/qx42ns9pyHTECXl22ARq77lYwwezzintZh8Y6c+sc0N8tM8NSg4D6DPHw10226rJZ7ZBOkSvOgBXPYCrgwE0WQXjYybiQAJuduvZh2MUhWbzoW5nOBDA9UOPbOOIzHsGG8os+XxIbDTRU3Ad2z4JecGFgL58gqa58e91BfJuoAY0Ps+ALfGHytEhKmcndPvYr5NzBW2K6rK93eYxJ9OiqtYngnLsj4m9gMtXxXo9w8/GEetblCtpchAwHCCIitW6SlGbJduqKNvZfADyOfaThzb5yeRp5NI4X3arpth2qwZYu3UrEG2qVbtJb2Af+z3E/cJFzTjOzQq0SJvP2Mi5mcKi5mNh9g1fwBvEn6MDFGSDtYoq7ZRp5Tr/B0gMDQAdMUqxHZuQtXXe7uqSI830Z7ZOoetPaBxQ/cMRQbWaJDhZksxQHBqoDfK1jCi+NunbmP4+ZW/YCW7iT7/yF5dpuVwD6c/OccIeOiCX1xy8AhdWCMN1Lo31k34ik79jZ2ZV1RuAIN6t01uwPM6O7HMTTlpb4OmkOstQ+Rx+9A9srAI7Lxkc3ibDowSlsH7tYa1Q3YN65yIM9RxF3YRfm2Y2PF64aLZvubBTULsZTMkZCCnGX8NxdKoiX4Nwn+n0OOuGDocMflMFIx9A8ZMBM8TLImtn3JqZAAnYF4gBNS1AX+R1Cmwp2z7DH+Dm4k+xc7Jfb4EkK7gFR82RLT405hSaAtUztr6z4vw8BgTGHWXGOkMiQb+h0ffsGYKamOWP2gttW+dHl1V1xY8AHmSqQ+A93RGwtKr2EmwuMFGyFowqipcGnCuZueYYNj9RCEttlDnfTLfbvFxyREGnXICpDYwLWwC7swTFlyeI3Iyzh3FdtJcJ27n4Xb3L59Kq+IwIXz5O2RoVKjtKCrlM1jIfnL2b/mQ/G+45sgZDnfK0tJIT/h7WsKmAj+cDaSAWzOc8vRMTZrhTn2TGzy9X/Y3Bz1/3vM4v0uw2WeflRXtpoIgTyC7jwVhzk6flzIaJs3gARjR1gnnJFSGc7z/EyhzPHYLmWyY05Ql0mDnEsKNgv9X5GkxvMBUzIMMFnIQzBvXUIT+eU1Z6+d3b53bkgy4BG54Zx2ndmfyqcalo8WWvwWH8HLGjOmVG9eHX1vyf9MOf7DkolPJ2SqHIeL4SZ5U6E9oN4rSqU+F47UpqMgNAaryfWQaIQIHa8kO8BJV+om7LyfkprupSWDuLzWa2/GBktCcQUFVAVDKJrfuAVkITL8Ew221QdRqUPAmCY00UcNfaLyxbN1KviYlPwMgBU2uX57/kg7GLRXXDxs0o+GeyqXKCw3pkLjkKEudSeVlkCbgE2VUDa/lYZJ3kZr7v29sSxElbZBrrRB0z0NY7cIduqaTZwWmHt9rf0osL9FrRVFjuahRO8DFfoLzK6pyq+c6hxgmERVmCoEqaPF/O7IBTeHOZxsKnNr8XavBFJyyBjV1D8lmpfDGWdbWFsxibNliylViOiR7nrDOnGVhQvMtyRgzPAFCsY8z+GDW4owUI1OSiTpdMcAlLWh1r2w8YDIYXH45OaMOGM5uxNxlVYPNTGARse+ScnFNQDBLKNWHGAV/Dxi8NCXKyLq5y+lKGTi3JuWKDYhfJBJXPWmIIPRjDTsw+GFf4bxh64A9GCVsn5BPbCPBQ6/Qip6ZG18L83X4SpqFkDXiZGvunkkYzxMFOQfPShBOQ122SrdH/YIC7JRh1W61jGn5J8ROGYGQ06FAuiNhQoDmeO0r32BLGLRsyO+r60H/n5rq6gFZzkWao65aC0zhcWNTAKzSRNZQYAQpVukGzO3qbJSgBmORPFoP9R+1lVbZ1tV6D9GuvqyM82CCPmrYuFjsk5olmel0gAAUEntAfDc0MPO3H7gUVUaDohB6SGdQ2wAU0HHfIk0xswj9yHpvE654JEMv0jh3CgGUjYEz1DMFxEQeEpNDEo43QAu8YZDrn+h3YglzTGhlgxsAD+Sj0yB8A5ruQb7bt7cS4Dq27Bu7hL0TlTO9Er37OYeL7GXWcDdMJvAHcPWyosCDFWAE/4sF7cVOVEiDIXg80kn7+nL1XNBD0vhvTP2o/HjO136xhUQ2LSIE+TEESbYqmQaHEVOoxmAmM0cAivwQubMGe0BaA2QY8vzI3J9Z+B5pfT69KJYqslJs2bXdNrL/9vz+8++urd9++TN68ePtWFzsCx2S2Z1ck/brdLWCdCRxLIcaYrsyZbDcQ7QSOL/PUO5kl9wH0vjK0r0BhJix43WyqKzV4/e66Akm+zNG8Blmo/eXNz9p1VV+B32aAuN/V2rufXnz7g1ZsQKjCK2Re6pFsCvio7bbLFB22QbAa7OFtWjd59+IybS7XxaJ7/kdTld1D1fBQNtIdeolI9hsl5F1senDgvGY5Sj20Jr794c3P794yZ42unkYsYt1bhTlobmfl5pFv2WHkuN7CTqPFyo1I4Dur3CN5FtirwLJtbxE4xHVXqzxaWs6S5IGn883gZhoIRjgesb4MItvKAUoY5E6aLkN/GVp+FgR55jj+wnGDyMmdRb7yw9RZ+T5ZLbMFySPX87PUCZcCLBAubfI2QVLEOvSKoogQywc8vNCySZqtFq5tp24e5L6froKlG4aWZy2IZ0cLa5Eus8yNFoFvWQuAKplWDehAapMZ1L0V2hXJSV/PzesaPOykzW/aGU5vLnebbTNjcf68bHbgTqVNVhRMcRoF5Y2YGMD01XVSpiXXqM/19+9L3aB+PzBfrO/a1VEo47IsgG1aNi3H4zLm3IAmOfH8WReWBUUiIVkBT870eqHPMZGxGrhXCzRAYBX1bJ1uFsv0ZAUuVbqchc9si7j019xY6Pr85NJkXDpbqMf10rzMbzh+EsZZVW93zSxbXQiEKW9mKXXDUeq1cIhKyv6cHV+ytnfifb/BMSUvHNBlM6NLA7AgTaStR1mCiLPd6MXsVTyEOjtrzvQ6vdbPKQFqJABCOtPZ4YT34PXUZ3qzXRctPoGFptOki27oYBi2sMFATBxM3RLsiiGkMsPBZyfe+bmqFWaAhgmCJSmWNwZ+XoCNJz7n7PM8jmeW0YdshSwExLBbT1fw0UEm5DJhOfnAFdje4h6XW+U9z3JNbUGTg+0OngE/lY0Ykq6Li5K/TLIUxMH0+I9Fg5Y/7SHG/i9997IfhPZ8v2ksgqEPSUTbUJKD8qKfU+kzk7r5tinWuNPsZSc9Ey49ASYQkeblMC2Hv0LDUufhfErRSdCIzpNkbiJBy/ZY71d2TCNxDch4HYDS+dibpAFhnoEKoGdOP+8NeswhdLEnYAsmTydicNsaI8H6X+H0oq7dFmVJRwBZTnQ0nI3Vetdcyq7HeAWIEnTFJXdWsv5GAgUdQQrXzYmmP4eOPZ40UigjSlc3WNYE2odQkIYNJYzwuT/FyMixJBdOQcRuthiRZSiwB/38lE8FHv2MvwSgrrKR2PYJTl5WZRWcH/kw8yG/9mOGTdKqqrq4KEpwYlQRcAagr9IaJehtUmwucIbzES3E4HtlBboHfWfEGQHGsbyAPdCRykhKPqR/3IeRJJJwGsbaCeqgG1gLF4EMBdYmwCHoJj5DE4vusn7MqTI/PoBIDD18p58fK8jOR/vAJ+QH4jsQ773xqTGhUsESTmnEQjaj6uq6AUW3XmuYqG4BnaWpj4/NYodRQO7GoPZgYkaxRECcbNJtsq4yGt6I9Wy70w1hKuLEitvLxsZjATljkzGtBEApFecyK7OhPVeCOOEYcn8LwcSS+JydUXRlIQsENoplAgc67jhHaAygCU1tUMVX30pMToeaPLABQ5rZ2d6jc25QkUtza2jqMqq7cuScDqGLxHG/FFtxUvnCB5F7mNPgpnVMtR5Nccxk5gNxICL2DXKl8BXAOS/ALhdNMtNLjN+107OGua/uzfxrpD5gMD+yJ8fCZC2Y2mBJUJNEniuOz7bPbRawwYV2TdNYnCEjw0w4iPP7YBg9cGwa6HfHYs6QROA+LAlYBrTq4NpwGLhrbnHA6U2abQp0Owwk+lkqchi7QHFlsSjBFLaDHEa1XvL4NEt9nLYxvOJzw8QsRDWbn27Ye4z76BhdKS9m80msWlqlEMeE7lrbB8Lj2LPZyw17Gcei9cSeXuFUbrNLa1JQIlKN3vg/8Q+NUksOaJ/auEBRIJz0sqlqZFXm1YKmSxMRpIaDPorROWQOH2hQ7I5lM/hiYTOMYCgYsvY/AZr/h3/+WsJVCjYJQDs5tCR+mNyIxeEH6ScfejhuHBvWT8ImMAI/ZNOU256YrB/HYkAwkD0iuUQdSD43pwqIyYRapwIGj9nHrQje0xDhxuDCIEZxIcInVzFb5XCTuNZ4S6U/1wsvX798rRvyMg29rHYlfdcfGUMHKxrxpFUE3Wk3uo9jVbJCrbq+PeGylIVqlGAOdRc/XZ3wpxbcahSzV5J4RTJxsXAlns+s818NELbHzGxwTVBJomwDIdImquKSOs/yYtua1N8xRJjkQlhtMTfNhsAMag+KTswslQIkH0HzLUTrwKWUvFpZJBlwIFpUQjwrKMQ5sKjkC0pzUGyYMqVBF2ZBxK4xMuJjy6CGNhgqYL2iAZOwTETCHR/uS0v4M/XYZODoxvrbfM0MW2Y+UEOB5TuYhav99c/feNriFqY6BvX9EWwTmIMLwy5+A9YGM8tNfS75tItdsaY2BFqyd7m1zE8Q3tDPZfERrPB0DXodOrxiqX2DO6V/ZjrO+Bbp8bKD8T2CmIbeR2a6KdCwSfr33PQBzzieAjubxmhGoWAsaFGVgsictOkSQOR13BUrSNTH+i7eDUsCCthBh3SZU1a5EJP5YMEz1MgyR83F6R4sZobrYP5e/5KGa68GkVfo1xVOrPK0xeALrf3LWiwwaKTUOPet6ZBst0xnfYJJ7HSbI/vUM7qZLE0l9ptV63XWDCvZoJyRsKYZ7S7sNkCWPzNRN3xOmJYU+dZlE/PmzjDCePqRUHrUGmpipQgFlbrB0Wov6MFUcy/QbuYfZtac90ruyc5wArG5DHXBPY0u6jxf3k6RaBx4+OJEk3IJlLkY1Llhz0XImL0xeZZP1tpooIikMjK/nF+QYd0LCNX3vHc5EyrjMQU/I5FsEf9LNpGezps2ZpMBE2H9EJi5vC7sFNsY2tfA6fkMl25YBryWAPSkBfdoRnGEDjw9fY7kPcVx/4zhLWJH5kpZHbVnsIOoLtMWYIhcKVxWYCWo4CkWM58Ibu0LYmVNm9S7EkPbonuGFWoXKAMwoZZ17Qb/CwQrixWoKa5tmM6S4EtV6gJmX5tm0FZDzb0bNAWB2Qe+vLSETQMRVsG+lR+LuirP9O9ev3zxXfLTix/+hhbYIZGxIYS/v/7puz8nb7/9f69ovEvk6u5Y74zq51O+hSDrTAyYMtadIYaiDV+jLG8v0VJsuo0UhnoP4KIDkKAXwKDEsf4ub9ap9s7VDf3nBlxG94aY+l1lAi5RkJKaEuQUlwxNoJUudg7n+4S/fuWG0HBLO3Oe7Ut8NqiRYPNyGlB3fz439vcBHGEeXOj8fISYGDpCjMGa7wmAYQgAG6ZilCOlB0Rmxpv6WhwmahvKgfI7TcepSLk699CEBBeTATjTZYtTTM4Hf07jUZpRNlDFjCzmI0d2BijfG9SRw0SoWmIw3T+e4fadnJBzbhcwU934SGMjLMzEA6O/ntL4DXspWGIqmNmBPNUmw5r8ALPyVdXAPF3kMH8eyyKKqduuIqVON00MZ5e9Rq8DpBb1DOZYe8JvwWBcIP5UnhTL2ZatqTSop89G4TFeJv2dGVidsq94Iul1kv7qCcLY9BBYQWEzSG/To411jW2yBanPNX2CSEJXWN1FAU42HF0YD/BBmmdKEouXD1W0EkeJRXcBZvGBw+6fabHOhBE3hpEwOyaeMGfk8ORKpzupsRN+rv2PuEnC9OsxG071Eojh5akGnsViTa/p9DXS/E7M2L2k6+0VDN9lTOZspWLoAyjTl+VIdts9lLinIGFIWv68vyZnpDvoDSFasSTv0eFYPKwkiF0CuAFFtW7TmAVLOIgjAWJupgvgQxMNomFwgRfA3s8ce9Y6gGEMGG5u6H9hPAMnr2hvtVVarJFxlhV1R+t8nd4AwHVeo4sqdCnjSLw8J2pb2RGkF6MoOzQ0kJmUaMoOc0gP5IzPth/LayO7jhneony6q6s/NwYNws8c5zeW16ZSn0eDVNngpTytySpb+EE5zQYvJJucRaQGbkd3R2WECfcCRHvMI6NYsstx4sFMrKrqqnbVADpfMA3DUgoNT/Ie+otqYlEe2jtArHxw0D5/SA0dkI2XGTY0nILHdnB7xoSNx6qZtIHe8/mowImrJ/ByeOUWLo9H+4Qv10X4Opt3WCYmDxoucAKILN3lqrExGKnOcj+cBxWR0QVPVD/dQ/s9c3CIo2qwu2Z6zhsOrwd7TPXaaAyfVRrzdXwninTyg+CNy8jiOwhyB9CGRwPjs/NTsCfBAu1TZP3dk1q9eSJss8FZ3Fb701Kn3WwiEj6mylmBMebz+d3Jo1HwB0aMckn3pHz4SpWAPA+IKxnnLtwup+EMESaP1ZD5+MrJHZSI4ZMxnU+L92Ta9kzwQVTfT249pyk4ay2oMtDnYOvPsHp8XQyi0L1cGpb1T3PbwyGrFwGmz+LDofZXBO7kqLugyYcBBEoTH836cI7g1/lzLJWZszJodj1lfDAHmTUENj/F36OyafxB7VrkJb19N76oxsok7r9CZ5LBBbpBZbVuTF/l0vTpK3SmLV2gG8HqBn0sGtqS0wSR+dC56NW4hCdFR1KEFVje5XjNWWZ1OsU9XXY+lfdkXScShttBMbpUoTPaOn6/bWrglF02iVw2fj8t8gcL4PDG6dtBv0ztN7bSlMuaqywBg8AcFPFTi0b/oRIxRe3lK3rTvmHlUfTrD1jFrz7GXY1aKzBZArUad7rDWP2j9hLva6HLiJdyMG95vNqhN8krumilNgJMVyu8aEsvBF6XmpTMZLesH+Qt8ktijC+ZA3DK30lqYZSVwBRomQ8TwQ9RqOw4bIpytk+tTl0qHWHHtCQY2nJA++5uPI5uGc4A+hrQrY3ESAZ+EQc11bu7HHOXc/NvINAeg5LijHQ4N/gtGPr5fm/6ELiCFgDzBKEur8Wne29O8FDWXbGscd8urvWowNYK85kTcTVRr8U6xCwAZ+h/546auFCJxwokCZzwvohfSAcpMpxVuK/tRMxauePw7Q/vXv30zYuXr5K337/+2yt+04HmD/CXpHim0uX98pJ0uYSDbBkM7YRRkGf2+VLowtQW+soYZqcSDJ+kiyZhdaSYiY+7iMrknQsW1kg6szQeBDzuNnREcSKGTYr2NsFs7C5Fu53loZkUTFAi8u9eYXF6VkzQ2xrdJ0MY+/yvNBdL7FBbiGk2asMI82luvPzu2zddHjzB0HKHBi0pWCa0B6upEBOOihJovBPL/DNcRvKX4n9iJd1xI4KhXZ/Z/JhW3D9zhmD4zfB9UEQXCUgDKy8xezeIrx7RsKs0By+ceC1FKZFqG1hfzsKP6aKAbSnyhqkTYBtQONqPx38TAYzTru6U3rTHG+vNMf9Gpu4Wsknjm/TKIzMG2E7it0OIK2+s2uLumGt3UDR6UDQ8KKeaJa7PnLLv1KG8dHSB95mydVpshvFW/Ffg3XMUD0kSx3oCzA7WYKKfdHF1IEYsrt+YL+oLENVl+4a+n9HIOnygX0iU8rYZLRDHpE+F16riM50XrIBhybKamPGbHHh0xNJ3urh7qlw7pZfv+Tj6B0di/DxbXYzuRdCvKWLQJu9DKGWhM44h/eojHIgrAHIIxNkX2zDk5zRhxW/k3WT5ttVe0T94U1EKZaQXcZ+tZF+YJOU8jQ627KSjwJxIg4IAxZAonvxPAHdSdH7z4tvvdKO7xxR3n0w0xtM2AVxB1EzITeWrDPBiOrvkxTJINBDU0AnxjtcZdv0k7lRz//k9tPiuTwJDNKi10NgekIhEXbsohcaW919hrVpCpyOW7SYW+6EQzX9sL95/1Y1TardxsDNsYhC/R5ZfVdkOzMKVtgU6Udux0VJtg5qqhA9NWV0vKvDWJPjo5V9f4jeGbIF63HNBkGeCRO+/wgPSDaGLwXmUN9VKeeTzK+8KdUSqPAGOKrxyT2dpHezlebccDFgkmG1U1+B3o4PhiOmYhDJ6NOYOe0ydcjhIxFNGxC3Hy+j0qNLZJsN+qD0uQAkwHtiCHSARDshWgqxNGf8MYLmhE3o9Wdwo8Pywe/Rs33Oc4Ww09HBze32pQDrrz5JNbNMKJf3iRaYtPRI/NG35BcxjWq547uNAMkzLMT0ZJgl905cVpe36pvwMR8oU947vAOpYgekoMOBF5Msv7AGqxHHMsAM6JA0takwaEDMqlTsJAQzhWsNRF3kF1hvbvk89S6CjDcbQL6D2b25vbhWIyjI0zTJJ6HheFNhh5Eee5xlqqxURx8XbzOrryLZdz44iy3eIY4fD5jAKfc8nbsR/+lY5VDdExCVe4Ns2cYENgiAKBkBdNwi9aIyLF8qYWPagOfAdl9hh4B2ERGBHgUucEPqHQUjIAJjth344pkYEc/i+BfN4ruf7QwQdV5p7zFdw8Gj0MKPfJdWAqE7ol5fgvgEbjfrRInJU9mDposFK+wH/hbBRgesHxHZs2xkOy5uEfiMPkzYf86T6mNfrdEtHE1/qTidXS9WxEyzEswiwsWvZvhM5Ll/Vrx07TtS433XSgbOM/8invWLhQZBgmxzb8wLYNMvtVT3nQytAnuWKXWFq0didLu/LjPQfOvIxjZ4T+JHr+G7guVYURl+K2I7gWeITEvnKGbUHnSP158t3/U2NwEUENDFjJNtVGn3LC4jjh6FHXC+M3Cd6PnQSl9Euossl9hcjYPCApUZfvOvjG+8hYPDFCOiRR+Nvf5aR95B37yTk0ZMo5LXDz0Ze1w2jKHR9sCRQNNtP3Pkg8vlWz5qe4tH8p7HmAxpBybtgr1oOCcBKjhyVoahLSE0d7GJ/KVLfvd1f/gmdDb56ZCt/sPHWE3v9lkaFtt5Ap38+2rrEDX34xRn5ibhf0g7/j9bav0Py2oHnWJGgseOqE3mhbavRC/Gu887czzJAuJDuQ0c+phGUDigby+ME/mIu5O8yCsEjgviHxWGmMg0eAZbYl2qwrdDvmPKgXAMDeV+ugUznGl5o66qlaYa82rLvZ8aiFQzLs0TDptqVWGjdFMvcfGCyQQ31w0R78wwUATXNoGYOALFR7uDwTMNgIQdmG3rLyX1EtsF9TLbBOTDbIAh2WMKBWJ8z4RCFgZxwIHJsFKwgsC37R2IHlvTogL7px3oucSL5MbQjqTOGk3vINvEc3+l7234EkrWXiXYAsJ1ejxEM7FmB/EyCIHxwLgRQNGXtSDzf9JWEgmuGsiQMQM3s92CID+pOhRmagWIxO8qcbmje4xQ5PjFdJbviO6anZEaI6cihfJeY98RRXCtS8zEBYEmUZIupZAeAEp6zH6QdmUrkkQS26XvqyhU0HXLf0m1Ai/gKTB98ICUHpK7DNd39ITiCUXxvANKTNygcLMMx3f1YuoEZOYOFK7Fm21V5wrHNkOwFGQ2o70dmoOLkyGRxgS/3ByYw7RcorB2gPSUjGZqKJY5I3pOhC4miNEnomEqUkgxY3TedaD9EF9ahIBmpm4XbrZwuOBr71+0Eg9Sk75lK3J0MOB1kgsRCByYRPUkSOiIP9nmTiG6giCLqo4YuiZzuZ9jskdAKpg12FksAT6Ef7RyUvPMIsUZogJ1i+d1POETDDV1nDxp2GJJ+dHBYDpG40TAHGFgkcHtAo2Z3lO4MbTDl+wGHpS8B3zFoBzTYnfsQupbnWtZUuog2g2Uo/xyGhTMiJNDEifbQGUCPMLMtV0KcHDS146k+JZ3bCZ3hOyeIbDJaq+PKHHvQhK7tBaP5Ai+QeHeYx3YcfzSzZXsSdQ6a2Yq88WYTcPvu3k0bjArL2rvb5MEUsMBccmWowwUHdjQktTVxBkIQqO5DzxqwlTvab0L8PUQgoWw4ioXjdzj3Qw475yOWDTzflcXFBEcPX4WKmDtMzll+5I2PuS/NPJraisiY3Q87zSTw7LE488M9C3XswJY5YlQfYpNHV0T4B9VDhHDC3SDyI+IEyhk8oBpCMo/2FENYLglI6HniqNtP1RBPq3pa1dOqnlb1tKqnVf23r8r2XMeyQm7ueZ6akwDPJQw9K+C2AXH2gLKJH4HDIPyjQbqF+LYHVqXFvEfPOxxDz4vAfbIcZlMPEpm/hUo+cTziexEZm/vgUFskdEPfC6iJ70V75gG/03YD3+MrJ4OsjmeHWLPGLHkVe/Bo8X+TIdwGVWkPTlzoBugEMNtUJZkfhJKr9QCquGjShiKWMFgY0MMOw8ibrOexbdd2SCh8EvKw7J8TgYsJboVF1xOo1CaATgB+K4sUBIorOuKzyPLAlg85GoG6dNt3MAfk+9Qdce3hEhwLzF3PtqcycI7t+1jH6U0lXz0C//yIB4PIAyqQAhf8QSdk4+whUItEfuiNPGw8e74FrbBWho3j/vslm4/19cQNuXs5oEIYwCHF0uopAbAXbhQAq7tTbhjuGbj7gU2sqWNqPWVJh1lS1w8j39+TJfWRIx+SJGUQ70uSunclSfHeFXinLf+fHhf87rCWatvil19SbVHd0K8JSDVwXNtqq/GLl/S/48TnaqUV7W/Lnw7vVXF89mVMAU/lGXHekyKla1EBVDdq/3K5Zzxbu/JK0GFfnnY4ZpAYpnQ7MF3biybyiHQteUy61v6dXw4D3R16D095+paalHNdk7hKRsU3IyVh4tlmHy49NCECmrjPI5PwiyRErIDIV6CGATQbdLbvuqEXhaBaw3EM07cjYo+vGvnwFmwNPyQENGpkPzqM5noHxdFcy4wIKHfQH3Ti4CGBNMko3hNII1YYWqEFFPEDvMcVPgXSvtCqbB+Lb0NfcM/AcvdIGAV25HP+2tNoh/tGBoePdHxM9IGjEkTgBRDL+51RjETgZNjEm0glfKmrK2DHwSHoBIf7RJH/LIo4/4L7DwOK+L9vigxO9b+kVHdAoSeWeWKZ/yqW8T7LdRW+ZG/yrvDvYM2WF7guGJ9gDNteGKomCTjXge87nme7lh363p7GQbRObQwOH+hikUDkRg4BK8f63WmipyTB06qeQnz7LkLgf96+9zuXPMt7UIiPQXz0PQgM8WXg2NIYH8bx6otca7brtLnEIB799tBdvXrMly7tj+OJSfcG1gAZNfhHEdsXVbscfEmThPyBwbTeP4weEUyLHhNMCw8Mpu1ZzB0hteiemNrU3u4Jq4H77gQP/1YlrPhWi8oDU4mqeYOK7SAyXee3BdW+SJWxbWMezSajoJgH2hjtA8cjkWsPC/7ggBLLGQ8Lnch//BfzHBY/s12Y3nYi/IIXwCzwwwdVotkHVaIFThT6gSUIQJ4CaE+relrV06qeVvW0qqdV/ResygEvw6XmgwuGzvBrV3wiWsACcMmjG709jU50+Ei8TO5EXfP+ypN/Cf2Adq4c/Ay+zDeJfa7vq8BCM9fpvu/AJ/9+BnQI4NF9s8Pgyy08+c7R4EsfHtIY7Gu0Dx+Jl3kCcAo4AcMnqfS0qlGQ6H0Jf7/69f8DiCIFLQ==')).decode('utf-8'))
SOURCE_HASHES = {'route_probe.py': '7cd9071bc4671873f027c66e4e243e9771947dba8b4ebe83dae4cc7d129783bc', 'run_route_smoke.py': '1e1d4dd285340d2e8fac5aa069b76c78a079cd8848b8807fcb7a2c3fdd504763', 'train4_masks.json': '517129f0660ee14e5ac50e66fd3839b3c72fb02cf0781e3880c5d0eacf011a68'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
import zipfile
CONFIG=OUTPUT/'protocol.json'
cfg=dict(checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),
    dataset_json=str(COCO_JSON),output=str(OUTPUT),project_commit=PROJECT_COMMIT,
    source_sha256=SOURCE_HASHES,models_source_sha256='18026877836052c83e75c0d896adcc89822bb35616173c7e8b14b17c9a22e9d8',
    samples=json.loads((REPO/'train4_masks.json').read_text()),
    rtol=1e-5,atol=1e-6,route_epsilon=1e-8,optimizer_updates=0,
    policy='FP32 eval, strict math SDP, original calls unchanged; two independent T4 workers, no updates',
    route='Last-layer head means; raw decoder-Q slice times raw Q-spatial slice; no renormalization',
    test_used=False,dev_used=False,caption_quality_evaluated=False,
    data_audit_sha256='3ac9ff4eb2d5be2b51fbf38d06240c8c111565d38feeac73f6bc31ca023a17d0')
CONFIG.write_text(json.dumps(cfg,ensure_ascii=False,indent=2),encoding='utf-8')
for name,source in SOURCES.items():
    target=OUTPUT/'source'/name
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(source,encoding='utf-8')
try:
    run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','h5py','numpy','Pillow','tqdm'])
    run([sys.executable,'-u','run_route_smoke.py','prepare','--config',CONFIG],cwd=REPO)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1',
        '--nproc_per_node=2','--max_restarts=0','run_route_smoke.py','worker','--config',CONFIG],cwd=REPO)
    receipts=[json.loads((OUTPUT/f'complete_rank{rank}.json').read_text()) for rank in range(2)]
    assert all(r['status']=='INTERFACE_SMOKE_PASS' and r['optimizer_updates']==0 for r in receipts)
    assert receipts[0]['before_state_sha256']==receipts[1]['before_state_sha256']
    print('INTERFACE SMOKE COMPLETE: 0 updates; caption gain NOT evaluated.',flush=True)
finally:
    if LOG.is_file():shutil.copyfile(LOG,OUTPUT/'notebook.log')
    review=Path(str(OUTPUT)+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as archive:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file() and p.suffix in ('.json','.log','.py'):
                archive.write(p,p.relative_to(OUTPUT))
    print('SEND REVIEW ZIP:',review,flush=True)


Gửi **coco_routing_interface_smoke_review.zip**, kể cả khi test lỗi.
Nếu không thấy traceback gốc: `print(LOG.read_text(encoding='utf-8')[-16000:])`.
Không đổi tolerance hoặc tiếp tục sang training nếu FAIL. PASS chỉ xác nhận kỹ thuật trên batch nhỏ.
Không cần ảnh gốc, detector cache, UPT, VG hoặc full annotations. Không lưu checkpoint mới.
Đường dẫn khác giữa hai tài khoản: chỉ sửa bốn INPUT paths ở cell đầu, giữ nguyên nội dung dataset.